In [1]:
import os
import random
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification
from simpletransformers import t5

from sklearn.model_selection import train_test_split

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

import wandb

wandb.login()

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


In [2]:
run = wandb.init(
    project = 'NUM-Machine-Learning-Lab-3',
    config = {
        'num_train_epochs': 15,
        "train_batch_size": 16,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True
    }
)

In [3]:
df = pd.DataFrame()
df_names = glob("../Lab 2/processed_dfs/*.parquet")

search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)

df['prefix'] = "summarize"

print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())

100%|██████████| 12/12 [00:00<00:00, 17.70it/s]



Dataframe memory usage
Index               132
target_text     1966107
input_text     17616445
prefix           954492
dtype: int64
Dataframe shape: (14462, 3)

All search terms:
audio+classification
audio+deep+learning
audio+encoding
audio+fast+fourier
audio+fourier
audio+generation
audio+machine+learning
audio+prediction
audio+recognition
audio+representation
audio+restoration
audio+signal
                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text     prefix  
0  This report proposes state-of-the-art research...  summarize  
1  In this work, we show that a factored hybrid h...  summarize  
2  End-to-end neural TTS has shown improved perfo...  summarize  
3  

In [4]:
test_size = 0.2
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}")

Training instance count: 11570
Test instance count: 2892


In [5]:
model = t5.T5Model(model_name = "t5-small", model_type = "t5", args = run.config, use_cuda = True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
model.train_model(train_df)

INFO:simpletransformers.t5.t5_utils: Creating features from dataset file at cache_dir/


  0%|          | 0/11570 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_utils: Saving features into cached file cache_dir/t5-small_cached_12811570
INFO:simpletransformers.t5.t5_model: Training started


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 1 of 1:   0%|          | 0/1447 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_model: Training of t5-small model complete. Saved to outputs/.


(1447, 2.621162835119837)

In [7]:
results = model.eval_model(test_df)

INFO:simpletransformers.t5.t5_utils: Creating features from dataset file at cache_dir/


  0%|          | 0/2892 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_utils: Saving features into cached file cache_dir/t5-small_cached_1282892


Running Evaluation:   0%|          | 0/29 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_model:{'eval_loss': 2.207440261183114}


In [8]:
def return_pred(model, test_df):
    random_num = np.random.randint(0, len(test_df))
    
    title = test_df.iloc[random_num]['target_text']
    abstract = ["summarize: " + test_df.iloc[random_num]['input_text']]
    predicted_title = model.predict(abstract)

    print(f"Test dataframe index: {random_num}")
    print(f'Actual Title: {title}')
    print(f'Predicted Title: {predicted_title[0]}')

In [9]:
for _ in range(30):
    return_pred(model, test_df)

Generating outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Decoding outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Test dataframe index: 1880
Actual Title: ERANNs: Efficient Residual Audio Neural Networks for Audio Pattern Recognition
Predicted Title: A Novel Convolutional Neural Network for Audio Pattern Recognition


Generating outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Decoding outputs:   0%|          | 0/1 [00:00<?, ?it/s]

...

Decoding outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Test dataframe index: 650
Actual Title: A Comparison of Pooling Methods on LSTM Models for Rare Acoustic Event Classification
Predicted Title: A Study on Acoustic Event Classification and Acoustic Event Detection


Generating outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Decoding outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Test dataframe index: 2641
Actual Title: Periodicity Pitch Detection in Complex Harmonies on EEG Timeline Data
Predicted Title: A Comparison of Frequency and Response Spectrum for Acoustic Impulses in the auditory brains
